# Preprocessing and integration


In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.settings.set_figure_params(dpi=140, facecolor="white", frameon=False)
sc.settings.verbosity = 3


In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
SOURCE_DATA_ROOT = Path(
    os.environ.get("EPEN_SOURCE_DATA", REPO_ROOT.parent / "analysis" / "anndata")
).expanduser().resolve()
OUTPUT_ROOT = Path(
    os.environ.get("EPEN_OUTPUT_DIR", REPO_ROOT / "outputs")
).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SCRIPTS_DIR = REPO_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

def prefer_checkpoint(local_name, source_path):
    local_path = OUTPUT_ROOT / local_name
    return local_path if local_path.exists() else SOURCE_DATA_ROOT / source_path


In [ ]:
SOURCE_CHECKPOINTS = {
    "concatenated": "Gill_Wu_ZhangQ_ZangD_concatenated.h5ad",
    "qc": "2000HVG_rg_cc_mt/Gill_Wu_ZhangQ_ZangD.qc.h5ad",
    "norm_pca": "2000HVG_rg_cc_mt/Gill_Wu_ZhangQ_ZangD.norm_pca.h5ad",
    "harmony": "2000HVG_rg_cc_mt/Gill_Wu_ZhangQ_ZangD_cc_mt_inter.h5ad",
}
SOURCE_CHECKPOINTS


## Load and concatenate data

Four projects were included: PRJNA518131, PRJNA785181, PRJNA687170, and PRJNA1029546.


In [ ]:
def keep_accession_prefix(adata, col="sample", out_col=None):
    """
    In adata.obs[col], keep only the token before the first underscore.
    Example: 'SAMN13067553_cellbender_filter' -> 'SAMN13067553'
    If out_col is None, overwrite `col`; otherwise write to `out_col`.
    """
    import pandas as pd

    if col not in adata.obs.columns:
        raise KeyError(f"Column '{col}' not found in adata.obs")

    # ➜ Use plain Python strings, NOT pandas "string" dtype
    s = adata.obs[col].astype(str)

    # prefix before first '_'
    acc = s.str.extract(r'^([^_]+)')[0]

    target = col if out_col is None else out_col

    # Either keep as object or convert to *category* with normal strings
    adata.obs[target] = pd.Categorical(acc.astype(str))

    return adata


In [ ]:
RAW_MATRIX_ROOT = Path(os.environ.get("EPEN_RAW_MATRIX_ROOT", REPO_ROOT.parent / "mtx_output"))

project_paths = {
    "PRJNA518131": RAW_MATRIX_ROOT / "PRJNA518131/cellranger/mtx_conversions/combined_cellbender_filter_matrix.h5ad",
    "PRJNA785181": RAW_MATRIX_ROOT / "PRJNA785181/cellranger/mtx_conversions/combined_cellbender_filter_matrix.h5ad",
    "PRJNA687170": RAW_MATRIX_ROOT / "PRJNA687170/cellranger/mtx_conversions/combined_cellbender_filter_matrix.h5ad",
    "PRJNA1029546": RAW_MATRIX_ROOT / "PRJNA1029546/cellranger/mtx_conversions/combined_cellbender_filter_matrix.h5ad",
}

project_objects = {}
for project, path in project_paths.items():
    project_objects[project] = sc.read_h5ad(path)
    keep_accession_prefix(project_objects[project], col="sample")

# Keep only the ependymoma sample from PRJNA1029546.
project_objects["PRJNA1029546"] = project_objects["PRJNA1029546"][
    project_objects["PRJNA1029546"].obs["sample"] == "SAMN37873687"
].copy()

adata = ad.concat(
    list(project_objects.values()),
    join="outer",
    label="project",
    keys=list(project_objects),
    merge="first",
)
adata.write_h5ad(OUTPUT_ROOT / "00_concatenated.h5ad", compression="gzip")


### Load concatenated counts


In [ ]:
adata = sc.read_h5ad(prefer_checkpoint(
    "00_concatenated.h5ad", SOURCE_CHECKPOINTS["concatenated"]
))
adata


## Quality control and doublet removal

Cells were filtered using `min_genes=100`, mitochondrial percentage ≤40%, per-sample MAD 3, and Scrublet. Genes detected in fewer than three cells were removed.


In [ ]:
from qc_utils import apply_basic_filters, remove_doublets, stash_raw_counts_layer

stash_raw_counts_layer(adata, layer_name="counts")
adata = apply_basic_filters(
    adata,
    sample_key="sample",
    nmads=3,
    min_genes=100,
    min_cells=3,
)
adata = remove_doublets(adata, batch_key="sample")
adata.write_h5ad(OUTPUT_ROOT / "01_qc.h5ad", compression="gzip")


### Load QC checkpoint


In [ ]:
adata = sc.read_h5ad(prefer_checkpoint(
    "01_qc.h5ad", SOURCE_CHECKPOINTS["qc"]
))
adata


## Normalization, feature selection, and PCA

Shifted-log normalization was followed by batch-aware deviance feature selection (2,000 genes), cell-cycle scoring, covariate regression, and PCA (50 components).


In [ ]:
from qc_utils import normalize_and_hvg_deviance, run_pca, score_and_regress_cell_cycle

adata = normalize_and_hvg_deviance(
    adata,
    n_top_genes=2000,
    layer_name="log1p_norm",
    batch_key="sample",
)
adata = score_and_regress_cell_cycle(adata, layer_name="log1p_norm")
adata = run_pca(adata, layer_name="log1p_norm", n_pcs=50)
adata.write_h5ad(OUTPUT_ROOT / "02_norm_pca.h5ad", compression="gzip")


### Load normalization/PCA checkpoint


In [ ]:
adata = sc.read_h5ad(prefer_checkpoint(
    "02_norm_pca.h5ad", SOURCE_CHECKPOINTS["norm_pca"]
))
adata


## Harmony integration

Harmony was applied by sample, followed by a 15-neighbor graph and UMAP.


In [ ]:
sc.external.pp.harmony_integrate(adata, key="sample", basis="X_pca")
sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=15)
sc.tl.umap(adata)
adata.write_h5ad(OUTPUT_ROOT / "03_harmony.h5ad", compression="gzip")


### Load Harmony checkpoint


In [ ]:
adata = sc.read_h5ad(prefer_checkpoint(
    "03_harmony.h5ad", SOURCE_CHECKPOINTS["harmony"]
))
adata


## Batch execution

The same stages can be submitted with `scripts/run_processing.sbatch`.
